In [1]:
import pandas as pd

In [ ]:
DATA_DIR = 'data'
df = pd.read_parquet(f'{DATA_DIR}/hypoglycemia.parquet')

In [46]:
hypoglycemia_codes = ["LOINC/14749-6", "SNOMED/33747003", "LOINC/LP416145-3"]
# remove all df rows that have a code in hypoglycemia_codes
print(f"Before removing hypoglycemia codes: {len(df)} rows")
df = df[~df['code'].isin(hypoglycemia_codes)]
print(f"After removing hypoglycemia codes: {len(df)} rows")

Before removing hypoglycemia codes: 41288495 rows
After removing hypoglycemia codes: 41288495 rows


In [18]:
labels = pd.read_parquet('hypoglycemia_label2.parquet')
# rename 'value' to 'label'
labels.rename(columns={'value': 'label'}, inplace=True)
labels.head()

,patient_id,prediction_time,label,value_multi,start,Sample_time,Time
0,115967106,2023-01-09 10:03:00,True,3,2019-02-18 10:12:00,NaT,2023-01-09 10:03:00
1,115967181,2021-01-15 14:46:00,True,3,2021-01-09 23:59:00,NaT,2021-01-15 14:46:00
2,115967229,2021-10-04 07:22:00,True,3,2020-09-05 00:05:00,NaT,2021-10-04 07:22:00
3,115967241,2019-08-22 10:45:00,True,3,2019-08-17 23:59:00,NaT,2019-08-22 10:45:00
4,115967258,2006-11-21 06:29:00,True,3,2006-07-08 05:50:00,NaT,2006-11-21 06:29:00


In [19]:
# find positive fraction 
pos_labels = labels[labels['label'] == True]
print("Positive fraction: ", len(pos_labels) / len(labels))

Positive fraction:  0.05745393634840871


In [20]:
# find time difference between 'Time' and 'start' columns
labels['Time'] = pd.to_datetime(labels['Time'])
labels['visit_start'] = pd.to_datetime(labels['start'])
labels['visit_end'] = labels['Time'] - pd.Timedelta(hours=1)
labels.head()

,patient_id,prediction_time,label,value_multi,start,Sample_time,Time,visit_start,visit_end
0,115967106,2023-01-09 10:03:00,True,3,2019-02-18 10:12:00,NaT,2023-01-09 10:03:00,2019-02-18 10:12:00,2023-01-09 09:03:00
1,115967181,2021-01-15 14:46:00,True,3,2021-01-09 23:59:00,NaT,2021-01-15 14:46:00,2021-01-09 23:59:00,2021-01-15 13:46:00
2,115967229,2021-10-04 07:22:00,True,3,2020-09-05 00:05:00,NaT,2021-10-04 07:22:00,2020-09-05 00:05:00,2021-10-04 06:22:00
3,115967241,2019-08-22 10:45:00,True,3,2019-08-17 23:59:00,NaT,2019-08-22 10:45:00,2019-08-17 23:59:00,2019-08-22 09:45:00
4,115967258,2006-11-21 06:29:00,True,3,2006-07-08 05:50:00,NaT,2006-11-21 06:29:00,2006-07-08 05:50:00,2006-11-21 05:29:00


In [21]:
# drop all columns in df except patient_id,start_time, code, value, 
df = df[['patient_id', 'start', 'code', 'value']]
# rename 'start' to 'event_time'
df.rename(columns={'start': 'event_time'}, inplace=True)
# drop all columns in labels except patient_id, visit_start, label, visit_end
labels = labels[['patient_id', 'visit_start', 'label', 'visit_end']]

In [22]:
# merge my_df and my_labels on patient_id
merged_df = df.merge(labels, on='patient_id', how='inner')
# filter my_df to only include rows where start_time is between start and end
merged_df = merged_df[(merged_df['event_time'] >= merged_df['visit_start']) & (merged_df['event_time'] <= merged_df['visit_end'])]

In [23]:
# keep only patient_id, event_time, code, value
merged_df = merged_df[['patient_id', 'event_time', 'code', 'value']]

In [24]:
# sort by patient_id, event_time
merged_df = merged_df.sort_values(by=['patient_id', 'event_time'])
# keep only the last 300 rows for each patient_id
merged_df = merged_df.groupby('patient_id').tail(500)

In [25]:
# for the `value` in merged_df, if the value can be interpreted as a float, convert it to float with 2 decimal places
def convert_to_float(value):
    try:
        return round(float(value), 2)
    except:
        return value
merged_df['value'] = merged_df['value'].apply(convert_to_float)

In [27]:
# compute the min event_time for each patient, and then create a new column `relative_time_mins` which is the difference between event_time and the min event_time for that patient
merged_df['event_time'] = pd.to_datetime(merged_df['event_time'])
merged_df['min_event_time'] = merged_df.groupby('patient_id')['event_time'].transform('min')
merged_df['relative_time_mins'] = (merged_df['event_time'] - merged_df['min_event_time']).dt.total_seconds() / 60.0
# drop the min_event_time column
merged_df.drop(columns=['min_event_time'], inplace=True)
merged_df

,patient_id,event_time,code,value,relative_time_mins
336,115967095,2008-10-10 06:05:00,LOINC/2345-7,120.0,0.0
337,115967095,2008-10-10 06:05:00,LOINC/711-2,0.1,0.0
338,115967095,2008-10-10 06:05:00,LOINC/770-8,86.3,0.0
339,115967095,2008-10-10 06:05:00,LOINC/2160-0,0.6,0.0
340,115967095,2008-10-10 06:05:00,LOINC/751-8,19.25,0.0
...,...,...,...,...,...
41285991,121124990,2018-03-23 23:59:00,RxNorm/1115005,None,5782.0
41285992,121124990,2018-03-24 13:28:00,LOINC/LP173418-7,None,6591.0
41285993,121124990,2018-03-24 13:44:00,LOINC/LP173418-7,None,6607.0
41285994,121124990,2018-03-25 12:56:00,LOINC/LP173418-7,None,7999.0


In [ ]:
all_codes = pd.read_parquet('all_codes.parquet')

In [29]:
# join merged_df with all_codes on code
merged_df = merged_df.merge(all_codes, on='code', how='left')


In [30]:
# in 'description' column remove all occurrences of '#' or 'Bld Auto', or any text in parentheses eg '(foo)'
merged_df['description'] = merged_df['description'].str.replace('#', '', regex=False)
merged_df['description'] = merged_df['description'].str.replace('Bld Auto', '', regex=False)
merged_df['description'] = merged_df['description'].str.replace(r'\(.*?\)', '', regex=True)

In [31]:
# drop all nan values in the description column
merged_df = merged_df[merged_df['description'].notna()]
# strip all whitespace from the description column
merged_df['description'] = merged_df['description'].str.strip()

In [45]:
# find description column values which contain Glucose
glucose_codes = merged_df[merged_df['description'].str.contains('Glucose', case=False)]
glucose_codes

,patient_id,event_time,code,value,relative_time_mins,description,Text
0,115967095,2008-10-10 06:05:00,LOINC/2345-7,120.0,0.0,Glucose SerPl-mCnc,Glucose SerPl-mCnc: 120.0
97,115967095,2008-10-11 06:40:00,LOINC/2345-7,119.0,1475.0,Glucose SerPl-mCnc,Glucose SerPl-mCnc: 119.0
169,115967095,2008-10-12 06:20:00,LOINC/2345-7,126.0,2895.0,Glucose SerPl-mCnc,Glucose SerPl-mCnc: 126.0
275,115967095,2008-10-13 06:56:00,LOINC/2345-7,133.0,4371.0,Glucose SerPl-mCnc,Glucose SerPl-mCnc: 133.0
331,115967095,2008-10-13 23:00:00,RxNorm/727517,None,5335.0,50 ML glucose 500 MG/ML Prefilled Syringe,50 ML glucose 500 MG/ML Prefilled Syringe
...,...,...,...,...,...,...,...
1306225,121124984,2006-01-21 11:25:00,LOINC/2345-7,91.0,486605.0,Glucose SerPl-mCnc,Glucose SerPl-mCnc: 91.0
1306314,121124984,2006-04-20 11:15:00,LOINC/2345-7,94.0,614755.0,Glucose SerPl-mCnc,Glucose SerPl-mCnc: 94.0
1307281,121124990,2018-03-21 07:39:00,SNOMED/104686004,None,1922.0,"Glucose measurement, blood, test strip","Glucose measurement, blood, test strip"
1307282,121124990,2018-03-21 07:39:00,LOINC/2339-0,87.0,1922.0,Glucose Bld-mCnc,Glucose Bld-mCnc: 87.0


In [32]:
# create text column by concatenating the description and value columns ONLY if the value is not nan
# else, just use the description column
merged_df['Text'] = merged_df.apply(lambda x: x['description'] + ': ' + str(x['value']) if pd.notna(x['value']) else x['description'], axis=1)

In [33]:
def _condense_notes_per_csn(group: pd.DataFrame) -> str:
    """Helper to condense text notes for a patient_id group."""
    # For rows with the same time_difference, we want to combine them into a single entry
    group = group.groupby('relative_time_mins').agg({
        'Text': lambda x: ";".join(x.astype(str))
    }).reset_index()

    return "\n".join(
        f"{row['relative_time_mins']} - {row['Text']}"
        for _, row in group.astype({'Text': str, 'relative_time_mins': str}).iterrows()
    )
# group by patient_id and relative_time_mins, and then condense the text notes
condensed_df = merged_df.groupby('patient_id').apply(_condense_notes_per_csn).reset_index()
condensed_df.rename(columns={0: 'Text'}, inplace=True)
condensed_df.head()

/tmp/ipykernel_1841317/3955256315.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  condensed_df = merged_df.groupby('patient_id').apply(_condense_notes_per_csn).reset_index()


,patient_id,Text
0,115967095,0.0 - Glucose SerPl-mCnc: 120.0;Eosinophil: 0....
1,115967096,0.0 - Resp rate: 18.0;Body height: 64.17\n66.0...
2,115967097,0.0 - Transpl surg Note
3,115967098,0.0 - White blood cell count: 19.0;Red blood c...
4,115967100,0.0 - Blood potassium measurement: 4.0;Ca-I Se...


In [34]:
demographics = pd.read_parquet('hypoglycemia_demo.parquet')
# keep only the patient_id and age columns
demographics = demographics[['patient_id', 'gender', 'age']]
# demo_text = 'Gender: ' + gender + ', Age: ' + age
demographics['DemoText'] = demographics.apply(lambda x: 'Gender' + ': ' + str(x['gender']) + ', ' + 'Age' + ': ' + str(x['age']), axis=1)
demographics = demographics[['patient_id', 'DemoText']]

In [35]:
# merge condensed_df and demographics on patient_id
# if patient_id is not in condensed_df, then do not include it in the merged_df
condensed_df = condensed_df.merge(demographics, on='patient_id', how='inner')

In [36]:
condensed_df.head()

,patient_id,Text,DemoText
0,115967095,0.0 - Glucose SerPl-mCnc: 120.0;Eosinophil: 0....,"Gender: F, Age: 74"
1,115967096,0.0 - Resp rate: 18.0;Body height: 64.17\n66.0...,"Gender: F, Age: 68"
2,115967097,0.0 - Transpl surg Note,"Gender: F, Age: 49"
3,115967098,0.0 - White blood cell count: 19.0;Red blood c...,"Gender: F, Age: 53"
4,115967100,0.0 - Blood potassium measurement: 4.0;Ca-I Se...,"Gender: F, Age: 32"


In [37]:
# now concatenate the DemoText and Text columns into a single column called 'Text'
condensed_df['Text'] = condensed_df['DemoText'] + '\n' + condensed_df['Text']

In [38]:
# now merge condensed_df with labels on patient_id
condensed_df = condensed_df.merge(labels[['patient_id', 'label']], on='patient_id', how='inner')

In [40]:
condensed_df.head()

,patient_id,Text,DemoText,label
0,115967095,"Gender: F, Age: 74\n0.0 - Glucose SerPl-mCnc: ...","Gender: F, Age: 74",False
1,115967096,"Gender: F, Age: 68\n0.0 - Resp rate: 18.0;Body...","Gender: F, Age: 68",False
2,115967097,"Gender: F, Age: 49\n0.0 - Transpl surg Note","Gender: F, Age: 49",False
3,115967098,"Gender: F, Age: 53\n0.0 - White blood cell cou...","Gender: F, Age: 53",False
4,115967100,"Gender: F, Age: 32\n0.0 - Blood potassium meas...","Gender: F, Age: 32",False


In [41]:
# rename label to Label
condensed_df.rename(columns={"label": "Label"}, inplace=True)

In [43]:
# save to llm_full_hypoglycemia.parquet
condensed_df.to_parquet('textified_hypoglycemia_full.parquet', index=False)

In [44]:
# also, split the data into val and test sets, in a 70-30 split
from sklearn.model_selection import train_test_split
val_df, test_df = train_test_split(condensed_df, test_size=0.3, random_state=1234)
val_df.to_parquet('textified_hypoglycemia_val.parquet', index=False)
test_df.to_parquet('textified_hypoglycemia_test.parquet', index=False)